# Lesson 7: Why attention scores are scaled

Query-key dot products are divided by the square root of the head size before softmax. This lesson shows why: the variance of a dot product grows with the number of components, and large score gaps make softmax concentrate on only a few choices.

The matching [C lesson](../c/src/attention_scaling_main.c) and [Java lesson](../java/src/main/java/learning/karpathy/gpt1/AttentionScalingLesson.java) use deterministic examples so their intermediate values match. Everything here is small enough to run on a CPU.

## Step 1: Watch dot-product variance grow with head size

With independent, centered, unit-variance query and key components, the dot-product variance grows with the head size. Dividing by its square root brings the variance back to about 1.

To make the pattern exact and repeatable, fix the query to all +1 values and enumerate every possible key made of -1/+1 values. This isolates the key-side randomness and enumerates its full score distribution instead of estimating it from random samples.

In [ ]:
import math
import torch
import torch.nn.functional as F

head_sizes = (1, 2, 4, 8)
print("head size | raw score variance | variance after dividing by sqrt(head size)")

for head_size in head_sizes:
    patterns = torch.arange(1 << head_size)
    bit_positions = torch.arange(head_size)
    bits = (patterns[:, None] >> bit_positions) & 1
    keys = torch.where(bits.bool(), 1.0, -1.0).to(torch.float64)
    query = torch.ones(head_size, dtype=torch.float64)

    raw_scores = keys @ query
    scaled_scores = raw_scores / math.sqrt(head_size)
    raw_variance = raw_scores.var(unbiased=False).item()
    scaled_variance = scaled_scores.var(unbiased=False).item()
    print(f"{head_size:9d} | {raw_variance:18.4f} | {scaled_variance:39.4f}")

print("The raw variance grows with head size; scaling brings it back to 1.")

## Step 2: See how large logits sharpen softmax

Softmax converts scores into probabilities. Multiplying the same score vector by 8 (the square root of a head size of 64) makes its largest score dominate more strongly. The entropy gives another view: lower entropy means probability is concentrated among fewer choices. A saturated softmax can also produce small gradients for alternatives.

In [ ]:
example_logits = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5], dtype=torch.float64)
larger_logits = 8.0 * example_logits

original_probabilities = F.softmax(example_logits, dim=-1)
larger_probabilities = F.softmax(larger_logits, dim=-1)

def entropy(probabilities):
    return -(probabilities * probabilities.log()).sum().item()

print("Example logits:", example_logits.tolist())
print("Softmax at original scale:", original_probabilities.tolist())
print("Logits multiplied by 8:", larger_logits.tolist())
print("Softmax at larger scale:", larger_probabilities.tolist())
print(
    f"Largest probability: {original_probabilities.max().item():.4f} -> "
    f"{larger_probabilities.max().item():.4f}"
)
print(f"Entropy: {entropy(original_probabilities):.4f} -> {entropy(larger_probabilities):.4f} nats")

## Step 3: Scale query-key scores before softmax

Here a query of eight +1 components meets five keys chosen to produce scores 8, 4, 0, -4, and -8. The unscaled scores give almost all probability to the first key. Dividing by the square root of eight reduces the score gaps, so softmax can distribute probability more smoothly while keeping the same ranking.

In [ ]:
head_size = 8
query = torch.ones(head_size, dtype=torch.float64)
keys = torch.ones((5, head_size), dtype=torch.float64)

# Each successive key has two more -1 components, giving dot products 8, 4, 0, -4, -8.
for key_index in range(len(keys)):
    keys[key_index, : 2 * key_index] = -1.0

raw_scores = keys @ query
scaled_scores = raw_scores / math.sqrt(head_size)
raw_probabilities = F.softmax(raw_scores, dim=-1)
scaled_probabilities = F.softmax(scaled_scores, dim=-1)

print("Query:", query.tolist())
print("Keys:")
print(keys)
print("Raw dot-product scores:", raw_scores.tolist())
print("Raw softmax weights:", raw_probabilities.tolist())
print("Scores divided by sqrt(8):", scaled_scores.tolist())
print("Scaled softmax weights:", scaled_probabilities.tolist())
print(
    f"Largest attention weight: {raw_probabilities.max().item():.4f} -> "
    f"{scaled_probabilities.max().item():.4f}"
)

assert torch.equal(torch.argsort(raw_scores), torch.argsort(scaled_scores))
assert scaled_probabilities.max() < raw_probabilities.max()

## Compare and experiment

Run **Terminal → Run Task → Run tutorial lesson**, then choose lesson **7**. The C and Java versions print the same variance table, softmax comparisons, and eight-dimensional query-key example.

Try another set of logits or change the head sizes. Why does scaling preserve the ordering of scores while changing the probabilities? How would a very concentrated softmax affect learning? The scaling factor is a stabilizing design choice; it does not change which key has the largest dot product.

References: [PyTorch softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html) and the [original GPT tutorial notebook](../gpt_dev.ipynb), in the section following the first query-key example.